In [19]:
tabla ='qtioo10'
col_tabla = 'solopefec'

In [20]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2

In [21]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [22]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
# fecha= fecha.iloc[0, 0]
# print(fecha)

now = datetime.now()

# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"

# c1= text(query)
# connection2.execute(c1)

In [23]:
fecha = '01/05/23'
# fecha= '2023-05-01'

In [24]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

query0 = f"""
select
d1.INFOPEORICENASICOD,
d1.INFOPECENASICOD,
d1.INFOPESOLOPENUM,
d1.INFOPESECNUM,
d1.INFOPECPSCOD,
d1.INFOPECPSORICENASICOD,
d1.INFOPECPSCENASICOD,
d1.INFOPECPSAREHOSCOD,
d1.INFOPECPSSERVHOSCOD,

a1.solopefec as solopefec,
a1.SOLOPEACTMEDNUM,
a1.SOLOPEBUSPACSECNUM
from {tabla} d1
left outer join qtsop10 a1 on a1.SOLOPEORICENASICOD = d1.INFOPEORICENASICOD
and a1.SOLOPECENASICOD    = d1.INFOPECENASICOD
and a1.SOLOPENUM    = d1.INFOPESOLOPENUM

where a1.{col_tabla}>='{fecha}'
"""

base2 = pd.read_sql_query(query0,con=connection0)


connection0.close()

In [25]:
base2.shape

(46637, 12)

In [26]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46637 entries, 0 to 46636
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   infopeoricenasicod     46637 non-null  object        
 1   infopecenasicod        46637 non-null  object        
 2   infopesolopenum        46637 non-null  int64         
 3   infopesecnum           46637 non-null  int64         
 4   infopecpscod           46637 non-null  object        
 5   infopecpsoricenasicod  0 non-null      object        
 6   infopecpscenasicod     0 non-null      object        
 7   infopecpsarehoscod     0 non-null      object        
 8   infopecpsservhoscod    0 non-null      object        
 9   solopefec              46637 non-null  datetime64[ns]
 10  solopeactmednum        46637 non-null  int64         
 11  solopebuspacsecnum     46637 non-null  int64         
dtypes: datetime64[ns](1), int64(4), object(7)
memory usage: 4.3+

In [27]:
#CREAMOS LA TABLA TEMPORAL
base2.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

637

In [28]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46637 entries, 0 to 46636
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   infopeoricenasicod     46637 non-null  object        
 1   infopecenasicod        46637 non-null  object        
 2   infopesolopenum        46637 non-null  int64         
 3   infopesecnum           46637 non-null  int64         
 4   infopecpscod           46637 non-null  object        
 5   infopecpsoricenasicod  0 non-null      object        
 6   infopecpscenasicod     0 non-null      object        
 7   infopecpsarehoscod     0 non-null      object        
 8   infopecpsservhoscod    0 non-null      object        
 9   solopefec              46637 non-null  datetime64[ns]
 10  solopeactmednum        46637 non-null  int64         
 11  solopebuspacsecnum     46637 non-null  int64         
dtypes: datetime64[ns](1), int64(4), object(7)
memory usage: 4.3+

In [29]:
#LA SUBIMOS AL POSTGRES SQL

query_count_before = text(f"SELECT COUNT(*) FROM {tabla}")
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")


Cantidad de filas en la tabla qtioo10 antes de la inserción: 1072255


In [30]:
#Borramos en caso el ETL se ejecute una segunda vez
borrando=text(f"DELETE FROM {tabla} WHERE {col_tabla} >='{fecha}'")
borrado = connection3.execute(borrando)

In [31]:
query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN infopeoricenasicod TYPE character(1),
ALTER COLUMN infopecenasicod TYPE character(3),
ALTER COLUMN infopesolopenum TYPE numeric(10,0),
ALTER COLUMN infopesecnum TYPE numeric(2,0),
ALTER COLUMN infopecpscod TYPE character(10),
ALTER COLUMN infopecpsoricenasicod TYPE character(1),
ALTER COLUMN infopecpscenasicod TYPE character(3),
ALTER COLUMN infopecpsarehoscod TYPE character(2),
ALTER COLUMN infopecpsservhoscod TYPE character(3),
ALTER COLUMN solopefec TYPE date,
ALTER COLUMN solopeactmednum TYPE numeric(10,0),
ALTER COLUMN solopebuspacsecnum TYPE numeric(10,0);

INSERT INTO {tabla} 
(infopeoricenasicod,infopecenasicod,infopesolopenum,infopesecnum,infopecpscod,infopecpsoricenasicod,infopecpscenasicod,infopecpsarehoscod,infopecpsservhoscod,solopefec,solopeactmednum,solopebuspacsecnum)
SELECT 
infopeoricenasicod,infopecenasicod,infopesolopenum,infopesecnum,infopecpscod,infopecpsoricenasicod,infopecpscenasicod,infopecpsarehoscod,infopecpsservhoscod,solopefec,solopeactmednum,solopebuspacsecnum
FROM tmp_{tabla};
"""

c1= text(query)
connection3.execute(c1)

In [32]:
query_count_after = text(f"SELECT COUNT(*) FROM {tabla}")
cant_despues = connection3.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla qtioo10 después de la inserción: 1072286
La cantidad de filas insertadas fue de: 31


In [33]:
#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
"""
c2= text(query2)
cursor=connection3.execute(c2)

In [34]:
connection1.close()
connection2.close()
connection3.close()

In [35]:
engine1.dispose()
engine2.dispose()
engine3.dispose()

AYUDA PARA EXTRAER COLUMNAS Y ESTRUCTURA DE TABLAS (NO ES PARTE DEL ETL)

In [36]:
import re

cadena = """
    infopeoricenasicod character(1) COLLATE pg_catalog."default",
    infopecenasicod character(3) COLLATE pg_catalog."default",
    infopesolopenum numeric(10,0),
    infopesecnum numeric(2,0),
    infopecpscod character(10) COLLATE pg_catalog."default",
    infopecpsoricenasicod character(1) COLLATE pg_catalog."default",
    infopecpscenasicod character(3) COLLATE pg_catalog."default",
    infopecpsarehoscod character(2) COLLATE pg_catalog."default",
    infopecpsservhoscod character(3) COLLATE pg_catalog."default",
    solopefec date,
    solopeactmednum numeric(10,0),
    solopebuspacsecnum numeric(10,0)
"""

# Reemplaza los caracteres innecesarios
cadena = cadena.replace(" COLLATE pg_catalog.\"default\",", "")
cadena = cadena.replace(" with time zone", "")

# Divide la cadena en una lista de líneas
lineas = cadena.split("\n")

# Crea la cadena de alteración de columnas
cadena_alter = ""
for i, linea in enumerate(lineas):
    palabras = linea.split()
    if len(palabras) >= 2:
        columna = palabras[0]
        tipo = palabras[1]
        if i == len(lineas) - 2:
            # Última línea, agrega punto y coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo};\n"
        else:
            # Otras líneas, agrega coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo},\n"

# Utiliza una expresión regular para eliminar las comas duplicadas
cadena_alter = re.sub(r',+$', ',', cadena_alter, flags=re.MULTILINE)

print(cadena_alter)



import re

nombrecitos = re.findall(r'ALTER COLUMN\s+(\S+)', cadena_alter)
insertado_col = ",".join(nombrecitos)

print(insertado_col)

ALTER COLUMN infopeoricenasicod TYPE character(1),
ALTER COLUMN infopecenasicod TYPE character(3),
ALTER COLUMN infopesolopenum TYPE numeric(10,0),
ALTER COLUMN infopesecnum TYPE numeric(2,0),
ALTER COLUMN infopecpscod TYPE character(10),
ALTER COLUMN infopecpsoricenasicod TYPE character(1),
ALTER COLUMN infopecpscenasicod TYPE character(3),
ALTER COLUMN infopecpsarehoscod TYPE character(2),
ALTER COLUMN infopecpsservhoscod TYPE character(3),
ALTER COLUMN solopefec TYPE date,
ALTER COLUMN solopeactmednum TYPE numeric(10,0),
ALTER COLUMN solopebuspacsecnum TYPE numeric(10,0);

infopeoricenasicod,infopecenasicod,infopesolopenum,infopesecnum,infopecpscod,infopecpsoricenasicod,infopecpscenasicod,infopecpsarehoscod,infopecpsservhoscod,solopefec,solopeactmednum,solopebuspacsecnum
